# YouTube Trending Content Analytics: Ireland vs. GB, US & India

**Business question:** Which content categories and markets should a platform like YouTube prioritize for creator investment, and how does Ireland's trending market compare to larger English-speaking markets?

**Approach:** This notebook analyzes 90 days of daily YouTube trending-chart snapshots across four markets (Ireland, Great Britain, United States, India), using a mix of exploratory data analysis and a predictive model to answer two questions:
1. **What drives how long a video stays trending**, and does this differ by market?
2. **Is Ireland's trending market structurally different** from larger markets, and if so, why does that matter for content strategy?

**Data source:** [Trending YouTube Videos, 113 Countries](https://www.kaggle.com/datasets/asaniczka/trending-youtube-videos-113-countries) (Kaggle, updated daily).

**Tools:** Python (pandas, scikit-learn, matplotlib/seaborn), Google Colab, Power BI (see `/dashboard` for the companion dashboard).

---


## 1. Setup and data acquisition

The source dataset is large (~2.7GB, 5.6M+ rows across 113 countries), so rather than downloading it manually, we pull it directly via the Kaggle API (`kagglehub`), which is faster and lets this notebook be re-run to get fresh data at any time.

**Note on reproducibility:** this dataset updates daily. Re-running this notebook later will pull the most recent 90-day window, which may shift exact figures slightly from what's reported in the written commentary below — this is expected, not an error.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub

# Requires a free Kaggle account + API token: https://www.kaggle.com/settings -> API
# os.environ['KAGGLE_API_TOKEN'] = 'YOUR_TOKEN_HERE'

path = kagglehub.dataset_download("asaniczka/trending-youtube-videos-113-countries")
df = pd.read_csv(f"{path}/trending_yt_videos_113_countries.csv")

print(df.shape)
df.head()

## 2. Scope the analysis

The full dataset spans 113 countries and nearly 3 years of daily snapshots — far more than needed for a focused analysis. We narrow to:
- **4 markets:** Ireland (IE), Great Britain (GB), United States (US), India (IN) — chosen to compare a small English-speaking market (Ireland) against larger ones, plus one large non-Western market for contrast.
- **Last 90 days** — recent enough to be current, long enough to capture real patterns (30 days risks noise; the full 3 years is unnecessary for this scope).

In [ ]:
df['snapshot_date'] = pd.to_datetime(df['snapshot_date'])
latest_date = df['snapshot_date'].max()
cutoff_date = latest_date - pd.Timedelta(days=90)

countries_to_check = ['IE', 'GB', 'US', 'IN']
df_filtered = df[
    (df['country'].isin(countries_to_check)) &
    (df['snapshot_date'] >= cutoff_date)
].copy()

print(df_filtered.shape)
df_filtered['country'].value_counts()

## 3. Data cleaning

Checking data types and missing values before any analysis. Text fields (`description`, `video_tags`, `language`) had missing values in a small-to-moderate share of rows (2-22%) — these are filled with clear placeholders rather than dropped, since dropping rows over a missing tag would bias the sample (e.g. certain content types tend not to use tags at all) and we lose no numeric data by keeping them.

In [ ]:
df_filtered['publish_date'] = pd.to_datetime(df_filtered['publish_date'])
df_filtered['description'] = df_filtered['description'].fillna('No description')
df_filtered['video_tags'] = df_filtered['video_tags'].fillna('No tags')
df_filtered['langauge'] = df_filtered['langauge'].fillna('Unknown')

print("Duplicate rows:", df_filtered.duplicated().sum())
print(df_filtered.isnull().sum())

## 4. Exploratory analysis: how do markets differ?

### 4.1 A misleading first look

A naive comparison of average engagement by country shows Ireland with dramatically higher average views than GB, US, or even India — counter-intuitive given Ireland's much smaller population. Comparing the **mean** against the **median** reveals why: Ireland's mean is being inflated by a small number of videos that stay trending (and accumulate views) for a very long time, each contributing many repeat rows to the dataset.

In [ ]:
print("MEAN by country:")
print(df_filtered.groupby('country')[['view_count', 'like_count', 'comment_count']].mean().round(0))
print()
print("MEDIAN by country:")
print(df_filtered.groupby('country')[['view_count', 'like_count', 'comment_count']].median().round(0))

### 4.2 One row per day, not one row per video

The dataset records one row per video *per day* it trends — so a video trending for three weeks appears as ~21 separate rows. This means a simple row count conflates "how much activity happened" with "how many distinct videos were involved." We check unique video counts per country to separate these:

In [ ]:
df_filtered.groupby('country')['video_id'].nunique()

**Finding:** Ireland has only ~800 unique videos across 90 days, versus 2,500-3,700+ in the other three markets, despite similar total row counts. This means the *same* handful of videos are re-appearing on Ireland's chart day after day — a structurally less diverse trending chart.

### 4.3 Building a per-video summary table

To compare markets fairly (rather than being skewed by repeat-appearance rows), we build a deduplicated table with one row per unique video, summarizing its trending lifespan and peak performance.

In [ ]:
video_summary = df_filtered.groupby(['video_id', 'title', 'channel_name', 'country']).agg(
    days_trending=('snapshot_date', 'nunique'),
    best_rank=('daily_rank', 'min'),
    peak_views=('view_count', 'max'),
    peak_likes=('like_count', 'max')
).reset_index()

print(video_summary.shape)
video_summary.sort_values('days_trending', ascending=False).head(10)

### 4.4 Visualizing the country comparison

In [ ]:
avg_days = video_summary.groupby('country')['days_trending'].mean().sort_values(ascending=False)

plt.figure(figsize=(8,5))
avg_days.plot(kind='bar', color='steelblue')
plt.title('Average Days a Video Stays Trending, by Country')
plt.ylabel('Average Days Trending')
plt.xlabel('Country')
plt.xticks(rotation=0)
plt.show()

In [ ]:
unique_videos = video_summary.groupby('country')['video_id'].nunique().sort_values(ascending=False)

plt.figure(figsize=(8,5))
unique_videos.plot(kind='bar', color='darkorange')
plt.title('Number of Unique Videos Trending, by Country (90 Days)')
plt.ylabel('Unique Videos')
plt.xlabel('Country')
plt.xticks(rotation=0)
plt.show()

### 4.5 Checking whether this is one outlier or a genuine pattern

A boxplot of peak views by country tests whether Ireland's numbers are driven by one or two viral outliers, or reflect the *typical* video. If only the outliers (dots above the box) were elevated, we'd suspect a fluke. Instead, Ireland's entire box — including the median line — sits higher than the other three countries, indicating a genuine market-wide pattern, not just one skewed data point.

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(data=video_summary, x='country', y='peak_views')
plt.title('Distribution of Peak Views by Country')
plt.ylabel('Peak Views')
plt.xlabel('Country')
plt.yscale('log')
plt.show()

> **EDA summary:** Ireland's YouTube trending chart shows significantly lower content diversity than the UK, US, or India — only ~800 unique videos appeared over 90 days, versus 2,500-3,700+ elsewhere. Videos that do trend in Ireland stay on the chart much longer (~5.5 days on average vs ~1.2-1.8 elsewhere) and reach higher peak view counts, even outside of outliers. This suggests Ireland's relatively small population and content market make it easier for large, often internationally-driven content to dominate the chart for extended periods, potentially crowding out smaller or local Irish content. Notably, genuinely Irish content did still break through — tracks by Kingfishr and a CMAT collaboration both reached Ireland's top-10 longest-trending videos, showing local content can compete despite the market's smaller scale.

---

## 5. Content categorization

The dataset has no pre-labeled content category, so one is built using keyword pattern-matching on video titles. This was refined iteratively: an initial rule set left ~57% of videos unclassified ("Other"); after two refinement passes based on sampling what was actually in that bucket, "Other" was reduced to ~45%.

In [ ]:
def categorize_video(title):
    text = str(title).lower()
    if any(word in text for word in ['i spent', 'i built', 'i launched', 'i survived', 'in my backyard', '$', 'last to leave', 'for 24 hours', 'for 100 days']):
        return 'Challenge/Stunt'
    elif any(word in text for word in ['reacts', 'reaction', 'react to']):
        return 'Reaction'
    elif any(word in text for word in ['official video', 'official music video', 'vevo', 'lyrics', '(music video)', 'music vid', '#video', 'lyrical']):
        return 'Music'
    elif any(word in text for word in ['trailer', 'teaser']):
        return 'Movie/TV Trailer'
    elif any(word in text for word in ['highlights', ' vs ', ' at ', 'fifa', 'world cup', 'nba', 'nfl', 'football', 'match']):
        return 'Sports'
    elif any(word in text for word in ['gameplay', 'gaming', 'playthrough', 'minecraft', 'fortnite', 'roblox', 'gta', 'ark ', 'boss', 'speedrun', 'simulator', 'escape', 'forza', 'free fire', 'lifesteal', 'chess', 'iem ', 'brawl', 'esports']):
        return 'Gaming'
    elif any(word in text for word in ['news', 'breaking']):
        return 'News'
    elif any(word in text for word in ['podcast', 'interview']):
        return 'Podcast/Interview'
    else:
        return 'Other'

video_summary['category'] = video_summary['title'].apply(categorize_video)
video_summary['category'].value_counts()

> **Limitation, stated honestly:** content was categorized using keyword-based pattern matching on video titles. This reliably identifies distinct genres like trailers, gaming, and music, but ~45% of videos didn't match a clear keyword pattern and are grouped as "Other" — likely a mix of vlogs, comedy, lifestyle, and niche content without a consistent titling convention. A production system would likely use YouTube's own category API field or a trained text classifier for higher precision.

In [ ]:
category_by_country = pd.crosstab(video_summary['country'], video_summary['category'], normalize='index') * 100

category_by_country.plot(kind='bar', stacked=True, figsize=(10,6), colormap='tab10')
plt.title('Content Category Mix by Country (%)')
plt.ylabel('% of Trending Videos')
plt.xlabel('Country')
plt.legend(title='Category', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

**Finding:** India stands out with a far higher Gaming share (~24%) than the other three markets. GB and Ireland show fairly similar category *mixes*, suggesting Ireland's chart isn't categorically different from GB — just far less diverse in the *volume* of unique videos filling each category.

---

## 6. Predictive modeling: what drives trending duration?

**Business question:** can we predict how long a video will stay trending, based on early, observable signals? A model like this could inform decisions such as which videos to boost promotion behind, or which markets need targeted content strategy.

### 6.1 Preparing the data

Text columns (`country`, `category`) are one-hot encoded into binary columns so a model can use them. `drop_first=True` avoids redundant columns (e.g. if a video isn't GB, IE, or US, it must be IN).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

model_data = video_summary[['country', 'category', 'best_rank', 'peak_views', 'peak_likes', 'days_trending']].copy()
model_data = pd.get_dummies(model_data, columns=['country', 'category'], drop_first=True)

X = model_data.drop('days_trending', axis=1)
y = model_data['days_trending']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Train:", X_train.shape, " Test:", X_test.shape)

### 6.2 Training and comparing three models

Rather than assuming the most sophisticated algorithm wins by default, three models spanning a range of complexity are trained and compared on held-out test data:
- **Linear Regression** — simplest baseline, assumes straight-line relationships.
- **Random Forest** — handles non-linear relationships, harder to over-interpret coefficient-by-coefficient but gives feature importance.
- **XGBoost** — usually the strongest default for tabular data, though more sensitive to hyperparameter tuning.

In [ ]:
from xgboost import XGBRegressor

results = {}

for name, m in [
    ("Linear Regression", LinearRegression()),
    ("Random Forest", RandomForestRegressor(n_estimators=100, random_state=42)),
    ("XGBoost", XGBRegressor(n_estimators=100, random_state=42)),
]:
    m.fit(X_train, y_train)
    pred = m.predict(X_test)
    results[name] = {
        "model": m,
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
        "R2": r2_score(y_test, pred),
    }

comparison = pd.DataFrame({k: {kk: vv for kk, vv in v.items() if kk != "model"} for k, v in results.items()}).T
comparison = comparison.sort_values("R2", ascending=False)
comparison.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,5))
axes[0].bar(comparison.index, comparison['MAE'], color=['steelblue' if i==0 else 'gray' for i in range(len(comparison))])
axes[0].set_title('Mean Absolute Error (Lower = Better)')
axes[0].set_ylabel('Days')

axes[1].bar(comparison.index, comparison['R2'], color=['steelblue' if i==0 else 'gray' for i in range(len(comparison))])
axes[1].set_title('R² Score (Higher = Better)')

plt.tight_layout()
plt.show()

**Result:** Random Forest was the strongest performer (R²≈0.68, average error under half a day), ahead of both Linear Regression and a default-hyperparameter XGBoost. This is a useful, honest finding in itself — a more "advanced" algorithm doesn't automatically win, especially without tuning, on a relatively small (~10.6K row), low-dimensionality dataset like this one.

### 6.3 What actually drives the prediction?

In [ ]:
best_model_name = comparison.index[0]
best_model = results[best_model_name]["model"]

importances = pd.DataFrame({
    'feature': X_train.columns,
    'importance': best_model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(8,6))
plt.barh(importances['feature'], importances['importance'], color='steelblue')
plt.title(f'What Drives Trending Duration? ({best_model_name} Feature Importance)')
plt.xlabel('Importance')
plt.gca().invert_yaxis()
plt.show()

> **Modeling conclusion:** Random Forest predicted trending duration with strong accuracy (R²≈0.68, average error under half a day), outperforming both Linear Regression and XGBoost. The single strongest predictor was **whether a video was trending in Ireland specifically** — more influential than the video's own peak views or likes. This independently confirms the EDA finding: Ireland's small, low-diversity trending chart isn't just a surface-level pattern, but a genuinely predictive market characteristic. For a platform like YouTube, this suggests market-specific dynamics (chart size, content supply) may matter as much as, or more than, a video's raw popularity when forecasting its trending lifespan in a given market.
>
> **Caveat:** `peak_views`/`peak_likes` are partly "downstream" of `days_trending` (a video that trends longer naturally accumulates more views), so some of the model's accuracy reflects this relationship rather than purely independent, early-signal prediction. A stronger follow-up would restrict features to only `best_rank` and market/category — available before a video's full trajectory is known — to test how much predictive power holds up.

---

## 7. Exporting data for the Power BI dashboard

Cleaned, modeling-ready tables (excluding free-text fields like `title`/`description`, which caused CSV parsing issues in Power BI due to special characters/emojis) are exported for the companion dashboard in `/dashboard`.

In [ ]:
dashboard_data = video_summary[['video_id', 'country', 'category',
                                   'days_trending', 'best_rank', 'peak_views', 'peak_likes']].copy()
dashboard_data.to_csv('youtube_dashboard_clean.csv', index=False)

model_results_export = comparison.reset_index().rename(columns={'index': 'Model'})
model_results_export.to_csv('model_comparison.csv', index=False)

importances.to_csv('feature_importance.csv', index=False)

ie_top10 = video_summary[video_summary['country'] == 'IE'].sort_values(
    'days_trending', ascending=False
)[['title', 'channel_name', 'days_trending', 'category']].head(10)
ie_top10.to_csv('ie_top10_videos.csv', index=False)

print("Exported: youtube_dashboard_clean.csv, model_comparison.csv, feature_importance.csv, ie_top10_videos.csv")

## 8. Summary and next steps

**Key findings:**
1. Ireland's YouTube trending chart has far lower content diversity (~800 unique videos vs. 2,500-3,700+ elsewhere over 90 days), driven by videos that stay trending far longer on average.
2. This isn't caused by one viral outlier — Ireland's *typical* trending video reaches higher peak views than the typical video elsewhere.
3. A Random Forest model confirmed this independently: **market (specifically, being Ireland) was the single strongest predictor** of how long a video trends — ahead of the video's own views or likes.
4. Local Irish content can still break through despite this (e.g. Kingfishr, CMAT), suggesting the pattern is about market structure, not an absolute barrier.

**Possible next steps:** extend the analysis to more markets to test whether "small market → lower diversity, higher longevity" holds generally; incorporate YouTube's native category taxonomy instead of keyword rules; test whether targeted local-creator support measurably shifts Ireland's chart diversity over time.

See the companion **Power BI dashboard** (`/dashboard/youtube_analytics_dashboard.pbix`) for an interactive presentation of these findings.
